# ML System Design for Interviews & Production

**Track:** Bonus — Interview & Architecture  
**Prerequisites:** Tracks 1-7 (or equivalent experience)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–180 minutes

---

## Why This Notebook Exists

The **ML System Design interview** is the single most common reason AI engineer candidates fail at top companies. You can know PyTorch inside-out, but if you can't design a fraud detection pipeline end-to-end on a whiteboard you won't get hired.

This notebook teaches a repeatable **framework** for decomposing any ML system design problem, then walks through three complete designs that reference concepts from throughout this curriculum.

---

## The 7-Step ML System Design Framework

Use this framework for EVERY ML system design interview:

| Step | Question to Answer | Time |
|------|-------------------|------|
| 1. **Clarify** | What is the business goal? What are the constraints? | 3 min |
| 2. **Metrics** | How will we measure success? (online & offline) | 3 min |
| 3. **Data** | What data do we have? How do we label it? | 5 min |
| 4. **Feature Engineering** | What signals matter? Real-time vs batch? | 5 min |
| 5. **Model** | Which architecture? Why? Training strategy? | 5 min |
| 6. **Serving** | How to serve at scale? Latency requirements? | 5 min |
| 7. **Monitoring** | Drift detection? Feedback loops? Failure modes? | 4 min |

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  DATA LAYER  │────▶│ MODEL LAYER  │────▶│ SERVING LAYER│
│              │     │              │     │              │
│ • Sources    │     │ • Training   │     │ • API (NB26) │
│ • Features   │     │ • Validation │     │ • Caching    │
│   (NB11)     │     │   (NB13-12)  │     │   (NB38)     │
│ • Storage    │     │ • Registry   │     │ • Scaling    │
│   (NB06-05)  │     │   (NB15,13)  │     │   (NB29)     │
└──────────────┘     └──────────────┘     └──────────────┘
        │                    │                    │
        └────────────────────┴────────────────────┘
                    MONITORING LAYER
                    • Drift (NB35) • Safety (NB40)
                    • Cost (NB36)  • Eval (NB37)
```

## 📑 Table of Contents

* [Why This Notebook Exists](#why-this-notebook-exists)
* [The 7-Step ML System Design Framework](#the-7-step-ml-system-design-framework)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [Design 1: Real-Time Fraud Detection System](#design-1-real-time-fraud-detection-system)
  * [Step 1: Clarify](#step-1-clarify)
  * [Step 2: Metrics](#step-2-metrics)
  * [Step 3: Data](#step-3-data)
  * [Step 5: Model Architecture](#step-5-model-architecture)
  * [Step 6: Serving Architecture](#step-6-serving-architecture)
  * [Step 7: Monitoring (NB35)](#step-7-monitoring-nb35)
* [Design 2: Search & Recommendation System](#design-2-search-recommendation-system)
  * [Step 1: Clarify](#step-1-clarify)
  * [Architecture: The Funnel](#architecture-the-funnel)
  * [Key Insight: This IS the RAG Pattern](#key-insight-this-is-the-rag-pattern)
* [Design 3: LLM-Powered Customer Support System](#design-3-llm-powered-customer-support-system)
  * [Architecture](#architecture)
  * [Model Decisions](#model-decisions)
  * [Cost Architecture (NB36)](#cost-architecture-nb36)
* [Interview Answer Template](#interview-answer-template)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Baseline first](#q1-baseline-first)
  * [Q2: Feature Store necessity](#q2-feature-store-necessity)
  * [Q3: Model choice](#q3-model-choice)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: Design a Content Moderation System](#exercise-1-design-a-content-moderation-system)
  * [Exercise 2: Design an LLM-Powered Code Review Bot](#exercise-2-design-an-llm-powered-code-review-bot)
  * [Exercise 3: Design a Real-Time Pricing Engine](#exercise-3-design-a-real-time-pricing-engine)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors jump straight to "I'd use a transformer." Seniors start with the business problem, identify constraints (latency, cost, data availability), and work backward to the simplest model architecture that meets requirements.

**Mental Model:** ML System Design is like architecture for a building. You don't start with "I'll use marble floors" — you start with "How many people? What's the budget? Code regulations?" Then you design the structure.

**Common Junior Pitfall:** Over-engineering. Proposing a 7B-parameter transformer for a problem a logistic regression with 10 features solves perfectly. Always ask: "Does the simplest solution work?"

---

## Design 1: Real-Time Fraud Detection System

**Prompt:** *"Design a fraud detection system for a payment processor handling 10,000 transactions per second."*

### Step 1: Clarify
- Business goal: Block fraudulent transactions BEFORE money moves
- Constraint: **< 100ms latency** (customer is waiting at checkout)
- Scale: 10K TPS = 864M transactions/day
- False positive cost: legitimate customer blocked (bad UX)
- False negative cost: fraud goes through ($$$$ loss)

### Step 2: Metrics

| Metric | Type | Target | Why |
|--------|------|--------|-----|
| **Precision@95% recall** | Offline | > 90% | Catch 95% of fraud with < 10% false positives |
| **p99 latency** | Online | < 100ms | Customer waiting at checkout |
| **Fraud loss rate** | Business | < 0.1% of volume | Dollar amount saved |
| **False decline rate** | Business | < 2% | Customer experience |

### Step 3: Data

| Source | Features | Freshness |
|--------|----------|----------|
| Transaction stream | amount, merchant, category, timestamp | Real-time |
| User profile | account age, avg spend, location history | Batch (daily) |
| Device signals | IP, device fingerprint, browser | Real-time |
| Historical patterns | last 7 days velocity, unusual merchants | Feature store (NB11) |

In [ ]:
# Step 4: Feature Engineering for Fraud
import numpy as np
from datetime import datetime

class FraudFeatureService:
    """Production feature engineering for fraud detection.
    
    Key insight: Most features are AGGREGATIONS over time windows.
    These are computed by the Feature Store (NB11 - Feast) and served
    at inference time with < 10ms latency.
    """
    
    def compute_features(self, transaction, user_history):
        features = {}
        
        # --- Real-time features (computed on the fly) ---
        features['amount'] = transaction['amount']
        features['hour_of_day'] = transaction['timestamp'].hour
        features['is_weekend'] = int(transaction['timestamp'].weekday() >= 5)
        
        # --- Velocity features (from Feature Store, NB11) ---
        features['txn_count_1h'] = user_history.get('txn_count_1h', 0)
        features['txn_count_24h'] = user_history.get('txn_count_24h', 0)
        features['unique_merchants_1h'] = user_history.get('unique_merchants_1h', 0)
        
        # --- Deviation features (is this unusual for this user?) ---
        avg_amount = user_history.get('avg_amount_30d', 100)
        features['amount_deviation'] = (transaction['amount'] - avg_amount) / (avg_amount + 1e-6)
        
        # --- Geolocation features ---
        features['distance_from_home'] = user_history.get('distance_km', 0)
        features['new_device'] = int(transaction.get('device_id') not in user_history.get('known_devices', []))
        
        return features

# Demo
service = FraudFeatureService()
txn = {'amount': 5000, 'timestamp': datetime(2026, 3, 15, 3, 30), 'device_id': 'new_phone'}
history = {'txn_count_1h': 5, 'txn_count_24h': 12, 'unique_merchants_1h': 4,
           'avg_amount_30d': 85, 'distance_km': 3200, 'known_devices': ['old_phone']}

features = service.compute_features(txn, history)
print('Fraud detection features:')
for k, v in features.items():
    flag = '🚨' if k in ['amount_deviation', 'new_device', 'distance_from_home'] and (
        (k == 'amount_deviation' and v > 5) or (k == 'new_device' and v == 1) or
        (k == 'distance_from_home' and v > 1000)
    ) else '  '
    print(f'  {flag} {k}: {v}')
print('\n🚨 = High-risk signal (amount 58x normal, new device, 3200km from home at 3:30AM)')

### Step 5: Model Architecture

**Why NOT a deep learning model?**
- Latency constraint: < 100ms total, model must be < 10ms
- Interpretability: regulators require explanations for declined transactions (NB41 SHAP)
- Tabular data: tree-based models dominate tabular benchmarks

**Recommended: Two-Stage Model**

```
Stage 1: XGBoost (fast, <5ms)          → Score: 0-1 probability
  │ Score > 0.95: BLOCK immediately
  │ Score < 0.10: ALLOW immediately  
  │ 0.10 < Score < 0.95: → Stage 2
  │
Stage 2: Deep model + rules engine      → Final decision
  │ Neural network with user embeddings
  │ Business rules (first-time $5000+ = review)
  │ → BLOCK / ALLOW / SEND TO HUMAN REVIEW
```

### Step 6: Serving Architecture

```
Transaction Event (Kafka, NB10)
  │
  ├─→ Feature Store (Feast, NB11)  ─→ Get user features (<10ms)
  │
  ├─→ Model Server (FastAPI, NB26) ─→ XGBoost prediction (<5ms)
  │
  ├─→ Decision Engine              ─→ Block/Allow/Review
  │
  └─→ Logging (Langfuse, NB36)     ─→ Log decision + features for retraining
```

### Step 7: Monitoring (NB35)

| Monitor | What | Alert When |
|---------|------|------------|
| Feature drift | Statistical distribution shift | p-value < 0.01 (NB35 EvidentlyAI) |
| Prediction drift | Model output distribution | Mean score shifts > 10% |
| Feedback loop | Actual fraud rate vs predicted | Fraud rate increases |
| Latency | p99 response time | > 100ms |

---

## Design 2: Search & Recommendation System

**Prompt:** *"Design a product search and recommendation system for an e-commerce platform with 100M products and 50M daily active users."*

### Step 1: Clarify
- Two modes: **Search** (user types query) and **Recommendations** (homepage/product page)
- Scale: 100M products, 50M DAU, ~500M queries/day
- Latency: < 200ms for search results
- Business metric: Revenue per session (click-through → add-to-cart → purchase)

### Architecture: The Funnel

```
100M products
      │
   [RETRIEVAL] ──→ ~1000 candidates        (fast, approximate)
      │                                     • BM25 for text (sparse)
      │                                     • ANN for embeddings (dense, NB20)
      │                                     • Hybrid = both (NB21 RAG pattern!)
      │
   [RANKING] ──→ ~100 scored results        (slower, accurate)
      │                                     • Cross-encoder or deep model
      │                                     • Personalization features
      │
   [RE-RANKING] ──→ ~20 final results       (business logic)
      │                                     • Diversity (not all same brand)
      │                                     • Boost: promotions, new items
      │                                     • Filter: out-of-stock, geo-restricted
```

### Key Insight: This IS the RAG Pattern

| RAG Component (NB21) | Search System |
|----------------------|---------------|
| Document chunking | Product catalog indexing |
| Embedding + vector DB | Two-tower model + ANN index |
| Retrieval (top-k) | Candidate generation |
| Re-ranking | Scoring model |
| LLM generation | Result presentation |

In [ ]:
# Two-Tower Model for Retrieval
import torch
import torch.nn as nn

class TwoTowerModel(nn.Module):
    """Two-tower architecture for search/recommendation.
    
    Tower 1 (Query): Encodes user query / user profile
    Tower 2 (Item):  Encodes product features
    
    Similarity = cosine(query_embedding, item_embedding)
    
    This is the same idea behind:
    - Sentence-BERT (NB20): encode query + document separately
    - RAG retrieval (NB21): find similar document chunks
    - CLIP (NB22): text tower + image tower
    """
    
    def __init__(self, query_vocab_size=50000, item_features_dim=128, embed_dim=64):
        super().__init__()
        
        # Query tower: embed search text
        self.query_tower = nn.Sequential(
            nn.Embedding(query_vocab_size, 128),
            # In practice: average pooling over token embeddings
        )
        self.query_proj = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, embed_dim)
        )
        
        # Item tower: encode product features
        self.item_tower = nn.Sequential(
            nn.Linear(item_features_dim, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
    
    def encode_query(self, query_tokens):
        embedded = self.query_tower(query_tokens).mean(dim=1)  # Average pooling
        return nn.functional.normalize(self.query_proj(embedded), dim=-1)
    
    def encode_item(self, item_features):
        return nn.functional.normalize(self.item_tower(item_features), dim=-1)
    
    def forward(self, query_tokens, item_features):
        q_emb = self.encode_query(query_tokens)
        i_emb = self.encode_item(item_features)
        return torch.sum(q_emb * i_emb, dim=-1)  # Cosine similarity

model = TwoTowerModel()
query = torch.randint(0, 50000, (4, 10))   # Batch of 4 queries, 10 tokens each
items = torch.randn(4, 128)                 # 4 items, 128 features each

scores = model(query, items)
print(f'Two-Tower Search Model')
print(f'  Query encoding: {model.encode_query(query).shape}  (batch=4, embed=64)')
print(f'  Item encoding:  {model.encode_item(items).shape}   (batch=4, embed=64)')
print(f'  Similarity scores: {scores.detach().numpy().round(3)}')
print(f'\n  Key advantage: Item embeddings can be PRE-COMPUTED and stored in')
print(f'  a vector database (NB20). At query time, only the query tower runs.')
print(f'  This makes retrieval over 100M items feasible in <50ms.')

---
## Design 3: LLM-Powered Customer Support System

**Prompt:** *"Design an AI customer support system that handles 80% of support tickets automatically."*

### Architecture

```
Customer Message
      │
   [CLASSIFIER] ──→ Intent Detection (NB08 HuggingFace pipeline)
      │               • billing, technical, shipping, account
      │
   [ROUTER] ──→ Simple query? → RAG Pipeline (NB21)
      │         Complex query? → Agentic System (NB23)
      │         Sensitive?     → Human handoff
      │
   [RAG PIPELINE]
      │  • Retrieve from knowledge base (product docs, FAQs, policies)
      │  • Generate response with guardrails (NB40)
      │  • Include citations (faithfulness check, NB37)
      │
   [GUARDRAILS] (NB40)
      │  • No hallucinated refund promises
      │  • No disclosure of internal processes
      │  • Escalation triggers (anger detection, legal keywords)
      │
   [MONITORING] (NB35, NB36, NB37)
      │  • Customer satisfaction score
      │  • Automation rate (target: 80%)
      │  • Escalation rate
      │  • Cost per resolved ticket vs human agent
```

### Model Decisions

| Component | Model Choice | Why |
|-----------|-------------|-----|
| Intent classifier | Fine-tuned DistilBERT | Fast (<20ms), cheap, 95%+ accuracy |
| RAG retriever | Sentence-BERT embeddings | Domain-tuned for support docs |
| Response generator | GPT-4o-mini or Llama 3.1 8B | Cost-effective for simple responses |
| Complex reasoning | GPT-4o (via AI Gateway, NB38) | Falls back to powerful model |
| Guardrails | NeMo Guardrails (NB40) | Topic containment + safety |

### Cost Architecture (NB36)

| Tier | Model | Cost/Ticket | % of Tickets |
|------|-------|-------------|-------------|
| 1 | DistilBERT + RAG + GPT-4o-mini | $0.003 | 60% |
| 2 | RAG + GPT-4o | $0.02 | 20% |
| 3 | Human agent | $5.00 | 20% |
| **Blended** | | **$1.01** | vs $5.00 (pure human) = **80% savings** |

---
## Interview Answer Template

Use this template for any ML system design question:

```
"Great question. Let me walk through this systematically.

First, let me clarify the requirements:
  - [Business goal]
  - [Key constraints: latency, scale, cost]
  - [What does success look like?]

For metrics, I'd track:
  - [Offline metric for model quality]
  - [Online metric for business impact]

For the data pipeline:
  - [Data sources]
  - [Feature engineering approach]
  - [Real-time vs batch features]

For the model:
  - [Start with the simplest baseline]
  - [Why this architecture over alternatives]
  - [Training strategy]

For serving at scale:
  - [Architecture diagram]
  - [How to handle the latency constraint]
  - [Caching and optimization]

For monitoring:
  - [What can go wrong]
  - [How we detect it]
  - [Feedback loop for improvement]
```

---

## ✅ Knowledge Check

### Q1: Baseline first
You're asked to build a content recommendation system. What's the simplest non-ML baseline you should try first?

<details><summary>👉 Answer</summary>

"Most popular items" — show the same top-10 items to everyone based on click count. No ML needed. This is a surprisingly strong baseline for many recommendation problems and gives you a clear target to beat. If your ML model can't beat "show popular items," your features or model are wrong.
</details>

### Q2: Feature Store necessity
Why can't you just compute features inside your model server at inference time?

<details><summary>👉 Answer</summary>

Many critical features require historical aggregations ("number of transactions in the last 24 hours"). Computing this at inference time would require querying a database with millions of rows per request, adding 100ms+ latency. A Feature Store (NB11 Feast) pre-computes these aggregations and serves them in <10ms. Additionally, it ensures training-serving consistency (same feature values used for training are used for inference).
</details>

### Q3: Model choice
For a search ranking model with a 50ms latency budget, why would you choose XGBoost over a transformer?

<details><summary>👉 Answer</summary>

XGBoost inference on tabular features takes <1ms. A transformer with 100M+ parameters takes 20-50ms even on GPU. With a 50ms end-to-end budget (including network, feature lookup, etc.), you may only have 5-10ms for the model. Additionally, XGBoost handles tabular/structured features natively and often outperforms neural networks on tabular data.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Design a Content Moderation System
Design a system that detects and removes harmful content (hate speech, violence, CSAM) from a social media platform with 500M daily posts. Walk through all 7 steps.

### Exercise 2: Design an LLM-Powered Code Review Bot
Design a system that automatically reviews pull requests for security vulnerabilities, code quality, and performance issues. Consider: Which model? How to handle context (full repo vs diff only)? False positive management?

### Exercise 3: Design a Real-Time Pricing Engine
Design a dynamic pricing system for a ride-sharing app. Consider: surge pricing, demand prediction, geographic features, fairness constraints, and latency requirements.